# 🎙️ Venom Voice Tool

**Developer:** Venom

**How to run:** Enter your Gmail in **Step 1** → run it → then run **Step 2**.

Your studio link appears in about 2–3 minutes.

**Support / WhatsApp:** 0313-6194428

In [ ]:
#@title 🔐 Step 1 — Verify Your Access { display-mode: "form" }
EMAIL = "" #@param {type:"string"}
ACCESS_CHECK_URL = "https://script.google.com/macros/s/AKfycbxy18siqvOPd8et0bstFHufbNyWIfQWYNXb2HA4n5BEDvacFSoKfxTKvYN8prqV7zzUjw/exec"

import requests
from urllib.parse import quote

signed_email = None
try:
    from google.colab import auth
    auth.authenticate_user()
    import google.auth
    from google.auth.transport.requests import Request
    creds, _project = google.auth.default()
    if not creds.valid:
        creds.refresh(Request())
    _token = creds.token
    if not _token:
        creds.refresh(Request())
        _token = creds.token
    _userinfo_resp = requests.get('https://www.googleapis.com/oauth2/v3/userinfo', headers={'Authorization': f'Bearer {_token}'}, timeout=20)
    _userinfo = _userinfo_resp.json()
    signed_email = (_userinfo.get('email') or '').strip().lower()
    if not signed_email:
        raise ValueError('email not found')
except Exception:
    print('⚠️ Sign-in nahi ho saka — Google sign-in dialog mukammal nahi hua ya Gmail hasil nahi ho saka.')
    print('Step 1 dobara chalao aur sign-in dialog mein apna Gmail account select karke Allow karo.')
    raise SystemExit

print(f'Aap ka Gmail: {signed_email}')

typed_email = EMAIL.strip().lower()
if typed_email and typed_email != signed_email:
    print(f'ℹ️ Note: EMAIL box mein likha gaya email ({typed_email}) sign-in wale Gmail ({signed_email}) se alag hai — access check sign-in wale Gmail pe hi hoga.')

email = signed_email

if not ACCESS_CHECK_URL:
    print('⚠️ Access check wala link abhi set nahi hai.')
    print('Pehle is cell ki upar wali ACCESS_CHECK_URL line mein quotes ke andar apna Apps Script deploy link paste karo, phir ye cell dobara chalao.')
else:
    data = None
    try:
        resp = requests.get(f'{ACCESS_CHECK_URL}?email={quote(email)}', timeout=20)
        data = resp.json()
    except Exception:
        print('🌐 Network masla — access check ho hi nahi saka.')
        print('Apna internet connection check karo aur ye cell dobara chalao. (Ye inkaar nahi hai — sirf connection fail hua hai.)')
    if data is not None:
        if data.get('ok'):
            with open('/content/.venom_ok', 'w') as f:
                f.write(email)
            print('✅ Access mil gaya — ab Step 2 chalao.')
        else:
            print('⛔ Access nahi hai — ye Gmail list mein nahi hai (ya expiry khatam hai). Access ke liye WhatsApp pe rabta karo.')


In [ ]:
#@title 🚀 Step 2 — Launch Venom Voice Tool { display-mode: "form" }
import os
if not os.path.exists('/content/.venom_ok'):
    raise SystemExit('⛔ Pehle Step 1 mein apna Gmail check karo.')
!pip install -q omnivoice "gradio==5.50.0" soundfile pydub librosa accelerate
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — set Runtime > Change runtime type > T4 GPU")
print("✅ Install done.")
# Venom Voice Tool - original Gradio app built on open-source OmniVoice (k2-fsa)
# License note: OmniVoice is Apache-2.0. Only clone voices you have permission to use.
import os
import json
import shutil
import numpy as np
import soundfile as sf
import torch
import gradio as gr
from omnivoice import OmniVoice

try:
    from omnivoice import OmniVoiceGenerationConfig
except Exception:
    OmniVoiceGenerationConfig = None

try:
    from omnivoice.utils.lang_map import LANG_NAMES, lang_display_name
    ALL_LANGS = ["Auto"] + sorted(lang_display_name(n) for n in LANG_NAMES)
except Exception:
    ALL_LANGS = ["Auto", "English", "Urdu", "Hindi", "Arabic", "Spanish", "French", "German", "Chinese", "Japanese", "Korean", "Turkish", "Punjabi"]

print("Loading Venom Voice Tool model ...")
model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0" if torch.cuda.is_available() else "cpu",
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    load_asr=True,
)
SR = getattr(model, "sampling_rate", 24000)
print(f"Model loaded. Sample rate: {SR}")

VOICE_PRESETS = {
    "Default (Auto)": "",
    "Young Woman - Friendly": "female, young adult, friendly, natural",
    "Young Woman - Cheerful": "female, young adult, cheerful, energetic",
    "Female Storyteller - Warm": "female, middle-aged, warm storyteller, clear narration",
    "Teen Girl": "female, teen, youthful",
    "Elderly Woman": "female, elderly, gentle",
    "British Woman": "female, british accent",
    "Soft Whisper - Female": "female, soft whisper",
    "Young Man - Casual": "male, young adult, casual, natural",
    "Deep Narrator - Male": "male, deep voice, narrator, clear",
    "Male Storyteller": "male, middle-aged, warm storyteller",
    "Teen Boy": "male, teen, youthful",
    "Elderly Man": "male, elderly, gentle",
    "British Man": "male, british accent",
    "American Man": "male, american accent",
    "Indian Man": "male, indian accent",
    "Soft Whisper - Male": "male, soft whisper",
}

EMOTION_TAGS = [
    "[laughter]", "[sigh]", "[confirmation-en]", "[question-en]", "[question-ah]",
    "[question-oh]", "[surprise-ah]", "[surprise-oh]", "[surprise-wa]", "[dissatisfaction-hnn]",
]

NO_VOICE = "— None —"
SAVED_DIR = os.path.join(os.getcwd(), "saved_voices")
SAVED_INDEX = os.path.join(SAVED_DIR, "index.json")


def _save(audio_list, tag):
    if audio_list is None:
        raise gr.Error("No audio generated. Try shorter text.")
    arr = np.asarray(audio_list[0] if isinstance(audio_list, (list, tuple)) else audio_list)
    os.makedirs("/content/venom_audio", exist_ok=True)
    path = f"/content/venom_audio/Venom-Voice-{tag}.wav"
    sf.write(path, arr, SR)
    return path


def _generate(text, language, instruct, clone_prompt, steps, guidance, speed, duration=0, denoise=True, preprocess=True, postprocess=True):
    # Official OmniVoice demo style: explicit config (denoise / preprocess_prompt /
    # postprocess_output are real OmniVoiceGenerationConfig fields), speed only
    # when != 1.0, duration only when > 0 (duration overrides speed when set).
    cfg = None
    if OmniVoiceGenerationConfig is not None:
        try:
            cfg = OmniVoiceGenerationConfig(
                num_step=int(steps), guidance_scale=float(guidance),
                denoise=bool(denoise), preprocess_prompt=bool(preprocess), postprocess_output=bool(postprocess),
            )
        except TypeError:
            cfg = OmniVoiceGenerationConfig(num_step=int(steps), guidance_scale=float(guidance))
    kwargs = {"text": text.strip()}
    kwargs["language"] = language if (language and language != "Auto") else None
    if instruct:
        kwargs["instruct"] = instruct
    if clone_prompt is not None:
        kwargs["voice_clone_prompt"] = clone_prompt
    if cfg is not None:
        kwargs["generation_config"] = cfg
    if speed is not None and float(speed) != 1.0:
        kwargs["speed"] = float(speed)
    if duration is not None and float(duration) > 0:
        kwargs["duration"] = float(duration)
    return model.generate(**kwargs)


# ---------------- Saved voices (real storage on disk) ----------------

def _load_index():
    try:
        if os.path.exists(SAVED_INDEX):
            with open(SAVED_INDEX, "r", encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, dict):
                return data
    except Exception:
        pass
    return {}


def _write_index(index):
    os.makedirs(SAVED_DIR, exist_ok=True)
    with open(SAVED_INDEX, "w", encoding="utf-8") as f:
        json.dump(index, f, ensure_ascii=False, indent=1)


def _saved_choices():
    return [NO_VOICE] + sorted(_load_index().keys())


def _safe_name(name):
    keep = []
    for ch in (name or "").strip():
        if ch.isalnum() or ch in (" ", "-", "_"):
            keep.append(ch)
    return "".join(keep).strip()


def save_voice_fn(name, ref_audio, ref_text):
    try:
        name = _safe_name(name)
        if not name:
            raise gr.Error("Please type a voice name first (e.g. My Narrator).")
        if ref_audio is None:
            raise gr.Error("Please upload the reference voice clip first, then save it.")
        rt = ref_text.strip() if ref_text and ref_text.strip() else ""
        if not rt:
            # No transcript typed: transcribe the (trimmed) clip like the clone fix does.
            import soundfile as _sf
            _data, _sr = _sf.read(ref_audio)
            use_audio = ref_audio
            if len(_data) / float(_sr) > 10:
                use_audio = "/content/venom_ref_trim.wav"
                _sf.write(use_audio, _data[: int(10 * _sr)], _sr)
            try:
                rt = model.transcribe(use_audio).strip()
            except Exception:
                rt = ""
            if not rt:
                raise gr.Error("Could not auto-detect the reference text. Type the exact words of the clip in Reference Text, then save again.")
        os.makedirs(SAVED_DIR, exist_ok=True)
        dest = os.path.join(SAVED_DIR, name + ".wav")
        shutil.copyfile(ref_audio, dest)
        index = _load_index()
        index[name] = {"file": name + ".wav", "ref_text": rt}
        _write_index(index)
        choices = _saved_choices()
        gr.Info(f"Voice '{name}' saved.")
        return gr.update(choices=choices, value=name), gr.update(choices=choices, value=name)
    except gr.Error:
        raise
    except Exception as e:
        raise gr.Error(f"Save failed: {e}")


def delete_voice_fn(name):
    try:
        if not name or name == NO_VOICE:
            raise gr.Error("Please select a saved voice to delete.")
        index = _load_index()
        if name not in index:
            raise gr.Error("That saved voice was not found.")
        try:
            os.remove(os.path.join(SAVED_DIR, index[name].get("file", name + ".wav")))
        except Exception:
            pass
        del index[name]
        _write_index(index)
        choices = _saved_choices()
        gr.Info(f"Voice '{name}' deleted.")
        return gr.update(choices=choices, value=NO_VOICE), gr.update(choices=choices, value=NO_VOICE)
    except gr.Error:
        raise
    except Exception as e:
        raise gr.Error(f"Delete failed: {e}")


# ---------------- Generation ----------------

def tts_fn(text, language, saved_voice, preset, speed, duration, steps, guidance, denoise, preprocess, postprocess):
    if not text or not text.strip():
        raise gr.Error("Please enter some text first.")
    if saved_voice and saved_voice != NO_VOICE:
        # A saved voice was chosen: generate through the clone path with the
        # stored reference clip + its exact reference text.
        index = _load_index()
        entry = index.get(saved_voice)
        if not entry:
            raise gr.Error("That saved voice was not found. Please pick another one.")
        ref_path = os.path.join(SAVED_DIR, entry.get("file", saved_voice + ".wav"))
        if not os.path.exists(ref_path):
            raise gr.Error("The saved voice file is missing. Please save the voice again.")
        try:
            prompt = model.create_voice_clone_prompt(ref_audio=ref_path, ref_text=entry.get("ref_text", ""))
            audio = _generate(text, language, "", prompt, steps, guidance, speed, duration, denoise, preprocess, postprocess)
        except gr.Error:
            raise
        except Exception as e:
            raise gr.Error(f"Generation with the saved voice failed: {e}")
        return _save(audio, "TTS")
    instruct = VOICE_PRESETS.get(preset, "")
    audio = _generate(text, language, instruct, None, steps, guidance, speed, duration, denoise, preprocess, postprocess)
    return _save(audio, "TTS")


def clone_fn(text, language, ref_audio, ref_text, speed, duration, steps, guidance, denoise, preprocess, postprocess):
    if not text or not text.strip():
        raise gr.Error("Please enter the text to speak in the cloned voice.")
    if ref_audio is None:
        raise gr.Error("Please upload a 3-10 second reference voice clip.")
    # --- V4 CLONE FIX ---
    # The model internally speaks (ref_text + your text). If ref_text is wrong
    # or missing, it first speaks the wrong/other sentence, then your text.
    # Fix: ref_text must ALWAYS be the exact words spoken in the reference
    # clip, and audio + text must stay aligned.
    import soundfile as _sf
    _data, _sr = _sf.read(ref_audio)
    _dur = len(_data) / float(_sr)
    rt = ref_text.strip() if ref_text and ref_text.strip() else ""
    use_audio = ref_audio
    if rt:
        if _dur > 20:
            raise gr.Error("Reference clip is too long (over 20 sec). Please upload a short 3-10 second clip and write its exact words.")
    else:
        # No transcript given: trim clip to first 10 seconds, transcribe THAT
        # exact part with the built-in Whisper (load_asr=True), and show the
        # transcript back in the Reference Text box for checking/correction.
        if _dur > 10:
            _trim = _data[: int(10 * _sr)]
            use_audio = "/content/venom_ref_trim.wav"
            _sf.write(use_audio, _trim, _sr)
        try:
            rt = model.transcribe(use_audio).strip()
        except Exception as e:
            raise gr.Error("Auto-transcribe failed. Please type the exact words spoken in the reference clip in the Reference Text box, then Generate again.")
        if not rt:
            raise gr.Error("Could not hear clear speech in the reference clip. Please type its exact words in the Reference Text box.")
    prompt = model.create_voice_clone_prompt(ref_audio=use_audio, ref_text=rt)
    audio = _generate(text, language, "", prompt, steps, guidance, speed, duration, denoise, preprocess, postprocess)
    _p = _save(audio, "Clone")
    return _p, rt


def _append_tag(text, tag):
    text = text or ""
    if text and not text.endswith(" "):
        text += " "
    return text + tag


def _settings(prefix):
    speed = gr.Slider(0.5, 1.5, value=1.0, step=0.05, label="Speed", info="1.0 = normal. >1 faster, <1 slower.")
    dur = gr.Slider(0, 60, value=0, step=0.5, label="Duration (seconds)", info="Fixed duration to override speed (0 = off).")
    steps = gr.Slider(4, 64, value=32, step=1, label="Inference Steps (Speed vs Quality)", info="20 = Fast. Use 32+ for higher quality (a bit slower).")
    guid = gr.Slider(0.0, 4.0, value=2.0, step=0.1, label="Guidance Scale (CFG)", info="Higher = closer to reference/prompt.")
    denoise = gr.Checkbox(value=True, label="Denoise")
    pre = gr.Checkbox(value=True, label="Preprocess Prompt (silence removal + trims reference)")
    post = gr.Checkbox(value=True, label="Postprocess Output (removes long silences)")
    return speed, dur, steps, guid, denoise, pre, post


def _chip_rows(textbox):
    for row_start in (0, 5):
        with gr.Row():
            for tag in EMOTION_TAGS[row_start:row_start + 5]:
                btn = gr.Button(tag, size="sm", elem_classes=["chip-btn"])
                btn.click(lambda t, tag=tag: _append_tag(t, tag), inputs=[textbox], outputs=[textbox])


CSS = """
/* Venom red theme — compact small-size scale, left-anchored 1140px container. */
html, body {font-size: 12px !important; background: #0d0406 !important;}
body > *, #root, .app, main, gradio-app {background: #0d0406 !important;}
.gradio-container {max-width: 1140px !important; width: 100% !important; min-height: 100vh !important; margin-left: 28px !important; margin-right: auto !important; padding: 8px 0 !important; font-size: 12px !important; line-height: 1.35 !important; background: #0d0406 !important;}
.gradio-container .tabs, .gradio-container .tabitem, .gradio-container [role="tabpanel"], .gradio-container footer, .gradio-container .built-with {background: transparent !important;}
#venom-header, #venom-footer {background: transparent !important; border: none !important; box-shadow: none !important; padding: 0 !important;}
.gradio-container label, .gradio-container label span {font-size: 12px !important; font-weight: 600 !important;}
.gradio-container input, .gradio-container textarea, .gradio-container select {font-size: 12px !important;}
.gradio-container input::placeholder, .gradio-container textarea::placeholder {font-size: 12px !important;}
.gradio-container ul.options li {font-size: 12px !important;}
.gradio-container .md p, .gradio-container .prose p {font-size: 12px !important;}
.gradio-container .block-label {background: #2b0a0e !important; color: #e0a3a8 !important; border: 1px solid #5c141c !important; border-radius: 6px !important; font-size: 11px !important; font-weight: 600 !important; width: fit-content !important; padding: 2px 8px !important; box-shadow: none !important;}
.gradio-container .block-label svg, .gradio-container .block-label img {width: 11px !important; height: 11px !important;}
.gradio-container .block {background: #140507 !important; border: 1px solid #3d0f16 !important; padding: 8px 10px !important; border-radius: 10px !important;}
.gradio-container .form {gap: 6px !important;}
.gradio-container .row, .gradio-container .gr-row {gap: 8px !important;}
.gradio-container .tabs > .tab-nav {border-bottom: 1px solid #3d0f16 !important;}
.gradio-container .tabs > .tab-nav button {font-size: 12px !important; padding: 5px 10px !important; color: #e8cdd0 !important; background: transparent !important;}
.gradio-container .tabs > .tab-nav button svg, .gradio-container .tabs > .tab-nav button img {width: 11px !important; height: 11px !important;}
.gradio-container .tabs > .tab-nav button.selected {color: #ff4757 !important; border-bottom: 2px solid #e01e2b !important;}
.gradio-container .accordion, .gradio-container details {padding: 4px 8px !important;}
.gradio-container .accordion button, .gradio-container .accordion button span, .gradio-container details summary, .gradio-container details summary span {font-size: 12px !important;}
.gradio-container input[type="range"] {height: 4px !important;}
.gen-btn button {background: linear-gradient(90deg, #c11120, #ff3b47) !important; color: #ffffff !important; font-size: 14px !important; font-weight: 800 !important; border-radius: 8px !important; padding: 9px !important; min-height: 0 !important; border: none !important;}
.chip-btn button {background: #2b0a0e !important; border: 1px solid #5c141c !important; color: #ff6b74 !important; font-size: 11px !important; padding: 4px 2px !important; min-height: 0 !important; border-radius: 6px !important;}
#support-banner {background: linear-gradient(100deg, #3d0a10 0%, #20060a 55%, #12040a 100%) !important; border: 1px solid #5c141c !important; border-left: 4px solid #e01e2b !important; border-radius: 12px !important; padding: 10px 14px !important;}
#support-banner-text {background: transparent !important; border: none !important; box-shadow: none !important; padding: 0 !important;}
#wa-btn, .wa-btn {background: linear-gradient(90deg, #22c55e, #10b981) !important; color: #ffffff !important; font-weight: 700 !important; font-size: 12px !important; border-radius: 8px !important; border: none !important; padding: 8px 18px !important; width: fit-content !important; min-height: 0 !important; box-shadow: none !important;}
#wa-btn:hover, .wa-btn:hover {background: linear-gradient(90deg, #22c55e, #10b981) !important; color: #ffffff !important;}
.dark-btn button {background: #2b0a0e !important; border: 1px solid #5c141c !important; color: #ff6b74 !important; font-weight: 700 !important; font-size: 12px !important; border-radius: 6px !important; min-height: 0 !important;}
.gradio-container .upload-container > svg {width: 16px !important; height: 16px !important;}
.gradio-container .upload-container button svg, .gradio-container .source-selection svg, .gradio-container .source-selection button svg {width: 12px !important; height: 12px !important;}
/* Compact audio blocks: ~100px min height so outputs/inputs are not tall slabs. */
.audio-box {min-height: 100px !important;}
.audio-box .empty, .audio-box .player, .audio-box .upload-container {min-height: 100px !important;}
.footer {text-align: center; opacity: .85; font-size: 10px; padding: 8px 0 2px 0; color: #c9a3a6;}
.footer div {font-size: 10px !important;}
"""

HEADER = """
<div style="text-align:center; padding: 6px 0 2px 0;">
  <div style="font-size:26px; font-weight:800; color:#f43f4e; letter-spacing:.2px;">🎤 Venom Voice Tool</div>
  <div style="color:#c9a3a6; font-size:12px; margin-top:4px;">Advanced AI Voice Studio — Voice Cloning &amp; Text-to-Speech in 600+ Languages</div>
  <div style="margin-top:8px;">
    <span style="display:inline-block; border:1px solid #5c141c; background:#22070a; color:#ff6b74; border-radius:999px; padding:3px 9px; font-size:11px; margin:2px;">🌐 600+ Languages</span>
    <span style="display:inline-block; border:1px solid #5c141c; background:#22070a; color:#ff6b74; border-radius:999px; padding:3px 9px; font-size:11px; margin:2px;">🎭 Voice Cloning</span>
    <span style="display:inline-block; border:1px solid #5c141c; background:#22070a; color:#ff6b74; border-radius:999px; padding:3px 9px; font-size:11px; margin:2px;">🎙️ 17 Voice Presets</span>
    <span style="display:inline-block; border:1px solid #5c141c; background:#22070a; color:#ff6b74; border-radius:999px; padding:3px 9px; font-size:11px; margin:2px;">⚡ Fast Generation</span>
  </div>
</div>
"""

BANNER_TEXT = """
<div>
  <div style="font-size:15px; font-weight:800; color:#ffffff;">🚀 Need Access or Support?</div>
  <div style="font-size:12px; color:#f0d5d7; margin-top:7px;">✅ Voice Cloning &amp; TTS &nbsp;•&nbsp; ✅ 600+ Languages &nbsp;•&nbsp; ✅ Fast Support</div>
  <div style="font-size:12px; color:#dbb3b6; margin-top:4px;">Contact Venom Voice Tool for subscription, renewals, and help.</div>
</div>
"""

FOOTER = """
<div class="footer">
  <div>© 2026 Venom Voice Tool · Support: WhatsApp</div>
  <div style="margin-top:4px; opacity:.75;">Use voices responsibly — only clone voices you have permission to use.</div>
</div>
"""

def _audio_chip(text):
    # Deterministic dark label chip shown above audio components whose native
    # Gradio label is hidden (the red theme rendered native labels as solid-red
    # bars and class-based CSS never reached them live).
    return gr.HTML(
        "<div style=\"display:inline-block; width:fit-content; background:#1c070a; "
        "border:1px solid #5c141c; color:#e0a3a8; font-size:11px; padding:3px 9px; "
        "border-radius:4px 4px 0 0;\">♪ " + text + "</div>"
    )


theme = gr.themes.Soft(primary_hue="red", secondary_hue="rose", neutral_hue="slate")

with gr.Blocks(title="Venom Voice Tool", theme=theme, css=CSS, fill_width=True) as demo:
    gr.HTML(HEADER, elem_id="venom-header")
    with gr.Column(elem_id="support-banner"):
        gr.HTML(BANNER_TEXT, elem_id="support-banner-text")
        wa_btn = gr.Button("💬 WhatsApp Support", elem_id="wa-btn", elem_classes=["wa-btn"])
    with gr.Tabs():
        with gr.TabItem("Text to Speech"):
            with gr.Row():
                with gr.Column(scale=3):
                    t_text = gr.Textbox(label="Text to Synthesize", lines=4, placeholder="Type or paste your text here, then press Generate...")
                    gr.Markdown("**Add emotion / event tags:**")
                    _chip_rows(t_text)
                    with gr.Row():
                        t_saved = gr.Dropdown(_saved_choices(), value=NO_VOICE, label="Saved Voice (optional)")
                        t_preset = gr.Dropdown(list(VOICE_PRESETS.keys()), value="Default (Auto)", label="Voice Design")
                    t_lang = gr.Dropdown(ALL_LANGS, value="Auto", label="Language")
                    with gr.Accordion("Generation Settings (optional)", open=False):
                        t_speed, t_dur, t_steps, t_guid, t_denoise, t_pre, t_post = _settings("t")
                    t_btn = gr.Button("Generate Speech", variant="primary", elem_classes=["gen-btn"])
                with gr.Column(scale=2):
                    _audio_chip("Output Audio")
                    t_out = gr.Audio(label="Output Audio", type="filepath", show_label=False, elem_classes=["audio-box"])
                    gr.Markdown("Your generated audio will appear here.")
        with gr.TabItem("Voice Clone"):
            with gr.Row():
                with gr.Column(scale=3):
                    _audio_chip("Reference Voice (3–10 seconds)")
                    c_ref = gr.Audio(label="Reference Voice (3–10 seconds)", type="filepath", sources=["upload", "microphone"], show_label=False, elem_classes=["audio-box"])
                    c_reftext = gr.Textbox(label="Reference Text (optional)", lines=2, placeholder="What the reference says. Leave blank to auto-detect.")
                    gr.Markdown("**Clone ka golden rule:** Reference clip **3-10 second** ka rakho, aur Reference Text me **bilkul wahi jumla** likho jo clip me bola gaya hai — ek lafz bhi idhar udhar hua to model pehle koi aur jumla bol dega. Aur jo nayi voice banani hai, us me **ek word ki jagah poora jumla** likho.")
                    c_text = gr.Textbox(label="Text to Synthesize", lines=4, placeholder="Type or paste your text here...")
                    gr.Markdown("**Add emotion / event tags:**")
                    _chip_rows(c_text)
                    c_lang = gr.Dropdown(ALL_LANGS, value="Auto", label="Language")
                    with gr.Accordion("Generation Settings (optional)", open=False):
                        c_speed, c_dur, c_steps, c_guid, c_denoise, c_pre, c_post = _settings("c")
                    c_btn = gr.Button("Generate in Cloned Voice", variant="primary", elem_classes=["gen-btn"])
                    gr.Markdown("---")
                    gr.Markdown("**Save this voice (after a successful clone):**")
                    with gr.Row():
                        s_name = gr.Textbox(label="Voice name", lines=1, placeholder="e.g. My Narrator", scale=2)
                        s_save = gr.Button("💾 Save Voice", elem_classes=["dark-btn"], scale=1)
                    with gr.Row():
                        c_saved = gr.Dropdown(_saved_choices(), value=NO_VOICE, label="Saved voices", scale=2)
                        s_del = gr.Button("🗑 Delete", elem_classes=["dark-btn"], scale=1)
                with gr.Column(scale=2):
                    _audio_chip("Output Audio")
                    c_out = gr.Audio(label="Output Audio", type="filepath", show_label=False, elem_classes=["audio-box"])
                    gr.Markdown("Your cloned audio will appear here.")
    gr.HTML(FOOTER, elem_id="venom-footer")
    wa_btn.click(lambda: gr.Info("WhatsApp Support: please contact the Venom Voice Tool owner for the support number."), outputs=[])
    t_btn.click(tts_fn, [t_text, t_lang, t_saved, t_preset, t_speed, t_dur, t_steps, t_guid, t_denoise, t_pre, t_post], [t_out])
    c_btn.click(clone_fn, [c_text, c_lang, c_ref, c_reftext, c_speed, c_dur, c_steps, c_guid, c_denoise, c_pre, c_post], [c_out, c_reftext])
    s_save.click(save_voice_fn, [s_name, c_ref, c_reftext], [t_saved, c_saved])
    s_del.click(delete_voice_fn, [c_saved], [t_saved, c_saved])

if __name__ == "__main__":
    demo.queue().launch(share=True, debug=True, inline=False)
